# Solar Inverter Failure Prediction Pipeline (Colab GPU Version)

This notebook is designed to run the solar failure prediction model on Google Colab, leveraging GPU acceleration for training.

### Instructions:
1.  **Change Runtime**: Go to `Runtime` -> `Change runtime type` -> Select `T4 GPU` (or any available GPU).
2.  **Upload Data**: Run the 'Data Upload' cell to upload `Copy of 54-10-EC-8C-14-69.raws.csv`.

In [ ]:
# Install dependencies if needed
!pip install xgboost joblib pandas numpy scikit-learn -q

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import IsolationForest
import json
import os
import joblib
from google.colab import files

print("Libraries imported. GPU Support Check:")
try:
    # Check if GPU is available for XGBoost
    import torch
    gpu_available = torch.cuda.is_available()
    print(f"GPU Available (via torch check): {gpu_available}")
except:
    print("Could not check GPU via torch, will attempt GPU training anyway.")

## 1. Data Upload
Run this cell and upload the `Copy of 54-10-EC-8C-14-69.raws.csv` file.

In [ ]:
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
print(f"Uploaded {file_name}")

In [ ]:
def clean_data(df):
    if "dc_voltage" in df.columns:
        df = df[df["dc_voltage"] > 0]
    if "ac_power" in df.columns:
        df = df[df["ac_power"] >= 0]
    if "grid_frequency" in df.columns:
        df = df[(df["grid_frequency"] > 45) & (df["grid_frequency"] < 65)]
    
    internal_cols = ["ac_power", "dc_voltage", "dc_current", "inverter_temp", "grid_voltage", "grid_frequency"]
    for col in internal_cols:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].median())
            
    external_cols = ["ambient_temp", "cloud_cover", "rainfall", "wind_speed", "storm_probability"]
    for col in external_cols:
        if col in df.columns:
            df[col] = df[col].fillna(method="ffill")
            
    return df

def feature_engineering(df):
    if "dc_voltage" in df.columns and "dc_current" in df.columns:
        df["dc_power"] = df["dc_voltage"] * df["dc_current"]
    
    if "dc_power" in df.columns and "ac_power" in df.columns:
        df["power_loss"] = df["dc_power"] - df["ac_power"]
        df["efficiency"] = np.where(df["dc_power"] > 0, df["ac_power"] / df["dc_power"], 0)

    if "inverter_temp" in df.columns:
        if "ambient_temp" in df.columns:
            df["thermal_stress"] = df["inverter_temp"] - df["ambient_temp"]
        else:
            df["thermal_stress"] = df["inverter_temp"]

    if "grid_voltage" in df.columns:
        df["voltage_deviation"] = abs(df["grid_voltage"] - 230)
    if "grid_frequency" in df.columns:
        df["frequency_deviation"] = abs(df["grid_frequency"] - 50)

    external_available = all(col in df.columns for col in ["storm_probability", "wind_speed", "cloud_cover", "rainfall"])
    if external_available:
        df["storm_risk"] = df["storm_probability"] * df["wind_speed"]
        df["weather_severity"] = df["cloud_cover"] + df["rainfall"] + df["storm_probability"]
    else:
        cols_to_fix = ["storm_risk", "weather_severity", "storm_probability", "wind_speed", "cloud_cover", "rainfall", "ambient_temp"]
        for col in cols_to_fix:
            if col not in df.columns:
                df[col] = 0

    return df

In [ ]:
print(f"Processing {file_name}...")
df = pd.read_csv(file_name)

feature_map = {
    'inverters[0].pv1_power': 'ac_power',
    'inverters[0].pv1_voltage': 'dc_voltage',
    'inverters[0].pv2_voltage': 'dc_current',
    'inverters[0].temp': 'inverter_temp',
    'sensors[0].ambient_temp': 'ambient_temp',
    'meters[0].meter_active_power': 'grid_active_power',
    'meters[0].grid_frequency': 'grid_frequency'
}

existing_features = [col for col in feature_map.keys() if col in df.columns]
df_mapped = df[existing_features].copy()
df_mapped = df_mapped.rename(columns={col: feature_map[col] for col in existing_features})

external_columns = ["cloud_cover", "rainfall", "wind_speed", "storm_probability"]
for col in external_columns:
    if col not in df_mapped.columns:
        df_mapped[col] = np.random.uniform(0, 1, size=len(df_mapped))

if 'inverters[0].alarm_code' in df.columns:
    df_mapped['failure'] = (df['inverters[0].alarm_code'] != 0).astype(int)
else:
    df_mapped['failure'] = (df_mapped['inverter_temp'] > 65).astype(int)

df_cleaned = clean_data(df_mapped)
df_featured = feature_engineering(df_cleaned)
df_featured = df_featured.dropna()

X = df_featured.drop(['failure'], axis=1)
y = df_featured['failure']

print(f"Features prepared: {X.shape[1]} features, {X.shape[0]} rows.")

In [ ]:
print("Starting XGBoost Training with GPU...")

tscv = TimeSeriesSplit(n_splits=5)

# Note: 'gpu_hist' is for historical versions, 'cuda' or 'hist' with device parameter is modern.
# Colab usually supports 'gpu_hist'.
model = xgb.XGBClassifier(
    n_estimators=1000, # Increased for GPU
    max_depth=7,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    objective='binary:logistic',
    tree_method='gpu_hist',  # ENABLE GPU
    predictor='gpu_predictor',
    random_state=42
)

for train_index, test_index in tscv.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Final Model Accuracy: {accuracy * 100:.2f}%")
print(classification_report(y_test, y_pred))

In [ ]:
print("Fitting Anomaly Detection Layer...")
iso_forest = IsolationForest(contamination=0.02, random_state=42)
iso_forest.fit(X)

model.save_model('solar_failure_model.json')
joblib.dump(iso_forest, 'anomaly_model.pkl')

with open('feature_meta.json', 'w') as f:
    json.dump({
        'features': list(X.columns),
        'accuracy': accuracy,
        'mode_detection': {
            'external_columns': external_columns + ["ambient_temp"]
        }
    }, f)

print("Artifacts saved: solar_failure_model.json, anomaly_model.pkl, feature_meta.json")

# Download artifacts back to local machine
files.download('solar_failure_model.json')
files.download('anomaly_model.pkl')
files.download('feature_meta.json')
print("Triggered downloads for model artifacts.")